In [ ]:
import json
import psycopg2
from kafka import KafkaConsumer
from sentence_transformers import SentenceTransformer

KAFKA_TOPIC = 'transactions'
KAFKA_BOOTSTRAP_SERVERS = 'kafka_streaming_lab:9092'
DB_CONFIG = {
    "host": "pgvector_snoql_lab", 
    "database": "vectordb",
    "user": "user",
    "password": "password",
    "port": 5432
}

/home/coder/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
conn = psycopg2.connect(**DB_CONFIG)
cursor = conn.cursor()

cursor.execute("""
        TRUNCATE TABLE transactions;
    """)
conn.commit()
cursor.close()
conn.close()

In [ ]:
model = SentenceTransformer('sdadas/mmlw-retrieval-roberta-large') 

conn = psycopg2.connect(**DB_CONFIG)
cursor = conn.cursor()

try:
    print("Sprawdzam strukturę bazy...")
    cursor.execute("CREATE EXTENSION IF NOT EXISTS vector;")
    # Tworzymy tabelę
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS transactions (
            id SERIAL PRIMARY KEY,
            sender VARCHAR(50),
            receiver VARCHAR(50),
            amount NUMERIC(10, 2),
            timestamp TIMESTAMP,
            device_sender VARCHAR(20),
            device_receiver VARCHAR(20),
            title TEXT,
            title_embedding vector(1024)
        );
    """)
    conn.commit()
    print("Baza przygotowana.")
except Exception as e:
    print(f"Błąd przy przygotowaniu bazy: {e}")
    conn.rollback()

# Konsument Kafki
consumer = KafkaConsumer(
    KAFKA_TOPIC,
    bootstrap_servers=[KAFKA_BOOTSTRAP_SERVERS],
    value_deserializer=lambda v: json.loads(v.decode('utf-8')),
    auto_offset_reset='earliest'
)

print("Konsument uruchomiony, czekam na transakcje...")

try:
    for message in consumer:
        tx = message.value
        embedding = model.encode(tx['title']).tolist()
        
        insert_query = """
            INSERT INTO transactions (sender, receiver, amount, timestamp, device_sender, device_receiver, title, title_embedding)
            VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
        """
        
        cursor.execute(insert_query, (
            tx['sender'], 
            tx['receiver'], 
            tx['amount'], 
            tx['timestamp'], 
            tx['device_sender'], 
            tx['device_receiver'], 
            tx['title'], 
            embedding
        ))
        
        conn.commit()
        print(f"Zapisano transakcję: {tx['title']}")

except Exception as e:
    print(f"Błąd podczas pracy konsumenta: {e}")
    conn.rollback()
finally:
    cursor.close()
    conn.close()

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 3155.97it/s]


Sprawdzam strukturę bazy...
Baza przygotowana.
Konsument uruchomiony, czekam na transakcje...
Zapisano transakcję: rachunek
Zapisano transakcję: przelew
Zapisano transakcję: zakupy
Zapisano transakcję: pizza
Zapisano transakcję: jedzenie
Zapisano transakcję: zakupy
Zapisano transakcję: zwrot kosztow
Zapisano transakcję: oddaje dlug
Zapisano transakcję: rachunek
Zapisano transakcję: zakupy
Zapisano transakcję: pizza
Zapisano transakcję: kino
Zapisano transakcję: zakupy
Zapisano transakcję: rachunek
Zapisano transakcję: przelew
Zapisano transakcję: rachunek
Zapisano transakcję: jedzenie
Zapisano transakcję: zakupy
Zapisano transakcję: jedzenie
Zapisano transakcję: oddaje pieniadze
Zapisano transakcję: zwrot kosztow
Zapisano transakcję: rachunek
Zapisano transakcję: rachunek
Zapisano transakcję: kino
Zapisano transakcję: zakupy
Zapisano transakcję: przelew
Zapisano transakcję: transport
Zapisano transakcję: jedzenie
Zapisano transakcję: jedzenie
Zapisano transakcję: zwrot za obiad
Zapisan